# GIAC Cancer Subtype Classification
Workflow: Mount Drive → Pull GitHub → Install → Train → Evaluate

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Kiểm tra data_final
DATA_DIR  = '/content/drive/MyDrive/ĐATN_2025.2/data_final'
GRAPH_DIR = '/content/drive/MyDrive/ĐATN_2025.2/Heterogeneous_Graph'

print('── data_final ──')
for f in sorted(os.listdir(DATA_DIR)):
    size = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f'  {f:45s} {size:.1f} MB')

print('\n── Heterogeneous_Graph ──')
for f in sorted(os.listdir(GRAPH_DIR)):
    fpath = os.path.join(GRAPH_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  {f:45s} {size:.1f} MB')
    else:
        print(f'  {f}/')
        for ff in sorted(os.listdir(fpath)):
            size = os.path.getsize(os.path.join(fpath, ff)) / 1e6
            print(f'    {ff:43s} {size:.1f} MB')

## 2. Clone / Pull repo từ GitHub

In [ ]:
REPO_URL = 'https://github.com/maivanquan-00/giac_project.git'

import os
if os.path.exists('/content/giac_project'):
    %cd /content/giac_project
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}
    %cd /content/giac_project
!ls

## 3. Install dependencies

In [ ]:
import torch
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {TORCH} | CUDA: {CUDA}')

!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html -q
!pip install torch-geometric entmax gseapy pyyaml tqdm scikit-learn -q
print('Done!')

## 4. Kiểm tra GPU

In [ ]:
import torch
print(f'CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 5. Train

In [ ]:
%cd /content/giac_project
!python train.py --config configs/config.yaml

## 6. Evaluate + top-K gene per patient

In [ ]:
!python evaluate.py \
    --config configs/config.yaml \
    --checkpoint checkpoints/best_model.pt \
    --split test \
    --top_k 50

## 7. Xem patient report

In [ ]:
import pandas as pd
df = pd.read_csv('patient_features.csv')
print(f'Total: {len(df)} | Accuracy: {df["correct"].mean():.4f}')
display(df.head(10))

## 8. Lưu kết quả về Drive

In [ ]:
import shutil, os
OUT = '/content/drive/MyDrive/ĐATN_2025.2/GIAC_results'
os.makedirs(OUT, exist_ok=True)
shutil.copy('patient_features.csv', OUT)
shutil.copy('checkpoints/best_model.pt', OUT)
print(f'Saved to {OUT}')